In [3]:
import pandas as pd 
import numpy as np 

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# options.add_argument('--headless')   # اگر خواستی بدون پنجره اجرا شود، این خط را فعال کن

ستاپ تست اینکه اطلاعات دیتا صفحه با جاوا اسکریپت کار میکند یا نه 

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.tgju.org/profile/geram18/history"
r = requests.get(url)
soup = BeautifulSoup(r.text, "lxml")

# چک کن
tables = soup.find_all("table")
if tables:
    print("جدول پیدا شد → احتمالاً بدون جاوا اسکریپت کار می‌کند")
    print(tables[0].text[:200])  # چند خط اول رو ببین آیا قیمت واقعی هست؟
else:
    print("جدول نیست → نیاز به جاوا اسکریپت دارد → سلنیوم/پلی‌رایت لازم است")

جدول پیدا شد → احتمالاً بدون JS کار می‌کند



بازگشایی؟
کمترین؟
بیشترین؟
پایانی؟
میزان تغییر
درصد تغییر
تاریخ / میلادی
تاریخ / شمسی




180,999,000
180,999,000
190,348,000
187,804,000
6798000

3.76%

2026/02/08
1404/11/19


193,566,000
180,860


In [2]:
import requests
import pandas as pd

url = "https://www.tgju.org/profile/geram18/history"  # یا صفحه آرشیو مورد نظرت
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

r = requests.get(url, headers=headers)
dfs = pd.read_html(r.text)

print(f"تعداد جدول‌های پیدا شده: {len(dfs)}")

# همه جدول‌ها رو چک کن تا جدول اصلی تاریخچه رو پیدا کنی (معمولاً جدول با ستون‌های زیاد)
for i, df in enumerate(dfs):
    print(f"\n--- جدول {i} --- شکل: {df.shape}")
    print(df.head(3))          # ۳ ردیف اول هر جدول
    print(df.columns.tolist()) # ستون‌ها رو ببین

تعداد جدول‌های پیدا شده: 1

--- جدول 0 --- شکل: (30, 8)
   بازگشایی؟    کمترین؟   بیشترین؟    پایانی؟ میزان تغییر درصد تغییر  \
0  180999000  180999000  190348000  187804000     6798000      3.76%   
1  193566000  180860000  193566000  181006000    12574000      6.95%   
2  192347000  192287000  194619000  193580000     1226000      0.64%   

  تاریخ / میلادی تاریخ / شمسی  
0     2026/02/08   1404/11/19  
1     2026/02/07   1404/11/18  
2     2026/02/05   1404/11/16  
['بازگشایی؟', 'کمترین؟', 'بیشترین؟', 'پایانی؟', 'میزان تغییر', 'درصد تغییر', 'تاریخ / میلادی', 'تاریخ / شمسی']


C:\Users\Magellan\AppData\Local\Temp\ipykernel_16504\20962089.py:8: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(r.text)


In [11]:
import requests
import pandas as pd
import time
import random
import re
from tqdm import tqdm

# ----------------- تنظیمات مهم -----------------
START_FROM = 0          # ← از اینجا ادامه بده (1110 ردیف قبلاً داری)
EXISTING_CSV = "geram18_history_partial.csv"   # اگر قبلاً ذخیره کردی این رو بخون

MAX_RETRIES = 3
TIMEOUT = 25               # ← خیلی مهم - timeout رو بیشتر کردیم
MIN_SLEEP = 5.0
MAX_SLEEP = 11.0
# ------------------------------------------------

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": "https://www.tgju.org/profile/geram18/history",
})

BASE_URL = "https://api.tgju.org/v1/market/indicator/summary-table-data/geram18"

# اگر فایل قبلی داری، بخونش
try:
    df_existing = pd.read_csv(EXISTING_CSV)
    all_rows = df_existing.values.tolist()
    print(f"→ {len(all_rows)} ردیف از قبل لود شد")
except:
    all_rows = []
    print("فایل قبلی پیدا نشد → از صفر شروع")

draw = (START_FROM // 30) + 1
start = START_FROM
length = 30
total = 3328

with tqdm(total=total, initial=start, desc="دریافت ردیف‌ها", unit="row") as pbar:
    while start < total:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                params = {
                    "lang": "fa",
                    "draw": draw,
                    "columns[0][data]": "0",  # ← همه columns رو دوباره گذاشتم
                    # ... بقیه columns[0] تا columns[7] رو مثل قبل کامل کپی کن
                    # (برای کوتاه شدن اینجا ننوشتم - از کد قبلی کپی کن)
                    "start": start,
                    "length": length,
                    "convert_to_ad": "1",
                    "_": int(time.time() * 1000),
                }
                # ← همه columns[0] تا [7] رو اینجا کامل بذار (۸ تا بلوک)

                r = session.get(BASE_URL, params=params, timeout=TIMEOUT)
                r.raise_for_status()
                data = r.json()

                if str(data.get("draw")) != str(draw):
                    print(f"draw mismatch در attempt {attempt}")
                    continue

                page_rows = data.get("data", [])
                if not page_rows:
                    print("صفحه خالی")
                    break

                cleaned = []
                for row in page_rows:
                    clean_row = [re.sub(r'<[^>]+>', '', cell).replace(',', '').strip() if isinstance(cell, str) else cell
                                 for cell in row]
                    cleaned.append(clean_row)

                all_rows.extend(cleaned)
                pbar.update(len(cleaned))

                print(f"start={start:4d} | +{len(cleaned):2d} ردیف | کل={len(all_rows):4d}")

                # ذخیره موقت بعد از هر صفحه
                pd.DataFrame(all_rows).to_csv("geram18_history_partial.csv", index=False)

                break  # موفق بود → برو درخواست بعدی

            except Exception as e:
                print(f"Attempt {attempt}/{MAX_RETRIES} شکست خورد → {e}")
                if attempt == MAX_RETRIES:
                    print("→ همه تلاش‌ها شکست خورد. خروج.")
                    break

        else:
            # اگر همه retryها شکست خورد
            break

        start += length
        draw += 1
        time.sleep(random.uniform(MIN_SLEEP, MAX_SLEEP))

# آخر کار
df = pd.DataFrame(all_rows, columns=['بازگشایی','کمترین','بیشترین','پایانی','میزان تغییر','درصد تغییر','تاریخ میلادی','تاریخ شمسی'])
df.to_csv("geram18_history_complete.csv", index=False, encoding="utf-8-sig")
print(f"\nپایان → کل ردیف‌ها: {len(df)}")

فایل قبلی پیدا نشد → از صفر شروع


دریافت ردیف‌ها:   0%|          | 0/3328 [00:00<?, ?row/s]

دریافت ردیف‌ها:   1%|          | 30/3328 [00:02<03:40, 14.98row/s]

start=   0 | +30 ردیف | کل=  30


دریافت ردیف‌ها:   2%|▏         | 60/3328 [00:13<13:49,  3.94row/s]

...

start=3270 | +30 ردیف | کل=3300


دریافت ردیف‌ها: 100%|██████████| 3328/3328 [15:23<00:00,  2.87row/s]

start=3300 | +28 ردیف | کل=3328


دریافت ردیف‌ها: 100%|██████████| 3328/3328 [15:29<00:00,  3.58row/s]


پایان → کل ردیف‌ها: 3328


In [ ]:
# ┌──────────────────────────────────────────────────────────────┐
# │                  tgju_crawler_template.py                     │
# │     شبه‌کد ماژولار - برای هر item فقط identifier عوض کن     │
# └──────────────────────────────────────────────────────────────┘

import requests
import pandas as pd
import time
import random
import re
from tqdm import tqdm
from typing import Optional, Tuple, List, Dict


def build_session() -> requests.Session:
    """هدرهای پایه – می‌تونی اینجا User-Agent rotate کنی"""
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36",
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "X-Requested-With": "XMLHttpRequest",
    })
    return s


def get_api_config(identifier: str) -> Dict:
    """تنظیمات API بر اساس identifier – فقط اینجا تغییر می‌دی"""
    return {
        "base_url": f"https://api.tgju.org/v1/market/indicator/summary-table-data/{identifier}",
        "referer": f"https://www.tgju.org/profile/{identifier}/history",
        "columns_count": 8,                     # تعداد ستون‌ها (برای geram18 = 8)
        "default_length": 30,                   # تقریباً همیشه 30 است
        "expected_columns": [                   # نام ستون‌ها – اگر item جدید متفاوت بود تغییر بده
            'بازگشایی', 'کمترین', 'بیشترین', 'پایانی',
            'میزان تغییر', 'درصد تغییر', 'تاریخ میلادی', 'تاریخ شمسی'
        ]
    }


def build_base_params(config: Dict) -> Dict:
    """ساخت params ثابت (columns و فیلترهای دیگر)"""
    params = {
        "lang": "fa",
        "convert_to_ad": "1",
        "search": "",
        "order_col": "",
        "from": "",
        "to": "",
        # order_dir دوبار – سایت حساس است
        "order_dir": "asc",
        "order_dir": "",
    }

    # اضافه کردن تمام columns[0] تا columns[N-1]
    for i in range(config["columns_count"]):
        params[f"columns[{i}][data]"]   = str(i)
        params[f"columns[{i}][name]"]   = ""
        params[f"columns[{i}][searchable]"] = "true"
        params[f"columns[{i}][orderable]"]  = "true"
        params[f"columns[{i}][search][value]"] = ""
        params[f"columns[{i}][search][regex]"] = "false"

    return params


def fetch_total_and_length(
    session: requests.Session,
    base_url: str,
    base_params: Dict
) -> Tuple[int, int]:
    """درخواست اول فقط برای گرفتن recordsTotal و length"""
    params = base_params.copy()
    params["draw"] = 1
    params["start"] = 0
    params["_"] = int(time.time() * 1000)

    r = session.get(base_url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()

    total = data.get("recordsTotal", 0)
    length = data.get("length", 30)  # fallback

    if total == 0:
        raise ValueError("هیچ رکوردی پیدا نشد → identifier اشتباه است؟")

    return total, length


def fetch_one_page(
    session: requests.Session,
    base_url: str,
    base_params: Dict,
    draw: int,
    start: int,
    length: int,
    max_retries: int = 3
) -> List[List]:
    """گرفتن یک صفحه با retry"""
    params = base_params.copy()
    params["draw"] = draw
    params["start"] = start
    params["length"] = length
    params["_"] = int(time.time() * 1000)

    for attempt in range(1, max_retries + 1):
        try:
            r = session.get(base_url, params=params, timeout=25)
            r.raise_for_status()
            data = r.json()

            if str(data.get("draw")) != str(draw):
                raise ValueError(f"draw mismatch (ارسالی: {draw}، برگشتی: {data.get('draw')})")

            rows = data.get("data", [])
            if not rows:
                return []

            # تمیز کردن
            cleaned = [
                [re.sub(r'<[^>]+>', '', cell).replace(',', '').strip() if isinstance(cell, str) else cell
                 for cell in row]
                for row in rows
            ]
            return cleaned

        except Exception as e:
            print(f"تلاش {attempt} شکست خورد: {e}")
            time.sleep(random.uniform(3, 8) * attempt)  # backoff

    raise RuntimeError(f"پس از {max_retries} تلاش موفق نشد")


def main_crawler(
    identifier: str,
    start_from: int = 0,
    temp_file: Optional[str] = None,
    final_file: Optional[str] = None,
    sleep_min: float = 5.0,
    sleep_max: float = 15.0
):
    """تابع اصلی – فقط identifier رو بده"""
    if temp_file is None:
        temp_file = f"{identifier}_temp.csv"
    if final_file is None:
        final_file = f"{identifier}_full.csv"

    session = build_session()
    config = get_api_config(identifier)
    base_params = build_base_params(config)

    print(f"شروع کراول → {identifier}")
    print(f"URL پایه: {config['base_url']}")
    print(f"Referer:   {config['referer']}")

    # ۱. گرفتن تعداد کل رکوردها
    total, length = fetch_total_and_length(session, config["base_url"], base_params)
    print(f"کل رکوردها: {total:,} | هر صفحه: {length} ردیف")

    # ۲. لود داده قبلی اگر وجود داشته باشد
    all_data = []
    if start_from == 0:
        try:
            df_old = pd.read_csv(temp_file, encoding="utf-8-sig")
            all_data = df_old.values.tolist()
            print(f"→ {len(all_data)} ردیف از قبل لود شد")
            start_from = len(all_data)
        except FileNotFoundError:
            pass

    draw = (start_from // length) + 1
    start = start_from

    with tqdm(total=total, initial=start, desc=f"{identifier}", unit="row") as pbar:
        while start < total:
            page_data = fetch_one_page(
                session, config["base_url"], base_params, draw, start, length
            )

            if not page_data:
                print("صفحه خالی بازگشت → پایان")
                break

            all_data.extend(page_data)
            pbar.update(len(page_data))

            # ذخیره موقت بعد هر صفحه
            pd.DataFrame(all_data).to_csv(temp_file, index=False, encoding="utf-8-sig")

            start += length
            draw += 1
            time.sleep(random.uniform(sleep_min, sleep_max))

    # ذخیره نهایی
    columns = config["expected_columns"]
    df = pd.DataFrame(all_data, columns=columns[:len(columns)])  # در صورت تفاوت تعداد ستون
    df.to_csv(final_file, index=False, encoding="utf-8-sig")
    print(f"پایان – فایل: {final_file} | ردیف: {len(df):,}")


# ────────────────────────────────────────────────
#                 نحوه استفاده
# ────────────────────────────────────────────────

# مثال ۱: طلای ۱۸ عیار (از قبل)
# main_crawler("geram18")

# مثال ۲: دلار آزاد
# main_crawler("price_dollar_rl")

# مثال ۳: سکه بهار آزادی
# main_crawler("sekeb")

# مثال ۴: ادامه دادن از وسط (اگر قطع شد)
# main_crawler("geram18", start_from=1500)